In [1]:
pip install psycopg2

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 MB 163.8 kB/s eta 0:00:17
    --------------------------------------- 0.0/2.8 MB 163.4 kB/s eta 0:00:17
   - -------------------------------------- 0.1/2.8 MB 245.8 kB/s eta 0:00:11
   - -------------------------------------- 0.1/2.8 MB 344.8 kB/s eta 0:00:08
   -- ------------------------------------- 0.1/2.8 MB 387.0 kB/s eta 0:00:07
   -- ------------------------------------- 0.2/2.8 MB 497.6 kB/s eta 0:00:06
   --- ------------------------------------ 0.3/2.8 MB 582.4 kB/s eta 0:00:05
   ---- --------------------------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
print(os.getcwd(), os.path.exists('global-bundle.pem'))

c:\Projects\data_engineering\Saylani cloud data engineering\cloud-data-engineering\batch 05 - Haniya Maqsood\Section 7 -- Kafka\Kafka-redshift-live-crpyto-prices True


In [2]:
import psycopg2

conn = psycopg2.connect(
    host='crypto-1.choa0ii4sygd.eu-north-1.rds.amazonaws.com',
    port=5432, dbname='postgres', user='postgres',
    password='postgres',
    sslmode='verify-full', sslrootcert='global-bundle.pem',
    connect_timeout=30)

In [3]:
cur = conn.cursor()
cur.execute("""
CREATE TABLE IF NOT EXISTS crypto_prices (
  symbol          VARCHAR(20),
  price           DECIMAL(18,8),
  event_ts        TIMESTAMP,
  kafka_partition INT,
  kafka_offset    BIGINT
)""")
conn.commit()

In [ ]:
import json
import psycopg2
from psycopg2.extras import execute_values
from kafka import KafkaConsumer

conn = psycopg2.connect(
    host='crypto-1.choa0ii4sygd.eu-north-1.rds.amazonaws.com',
    port=5432, dbname='postgres', user='postgres',
    password='postgres',
    sslmode='verify-full', sslrootcert='global-bundle.pem',
    connect_timeout=30)
cur = conn.cursor()

consumer = KafkaConsumer(
    'crypto_price',
    bootstrap_servers=['16.171.10.142:9092'],
    group_id='crypto-loader',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
)

BATCH_SIZE = 10          # small for testing; use 50 later
batch = []
for msg in consumer:
    v = msg.value
    print("message value", v)
    batch.append((v['symbol'], v['price'], v['ts'], msg.partition, msg.offset))
    if len(batch) >= BATCH_SIZE:
        execute_values(cur,
            'INSERT INTO crypto_prices (symbol, price, event_ts, kafka_partition, kafka_offset) VALUES %s',
            batch)
        conn.commit()
        consumer.commit()
        print('loaded', len(batch), 'rows')
        batch.clear()

C:\Users\HANIYA MAQSOOD\AppData\Local\Temp\ipykernel_33800\3202113153.py:14: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


message value {'symbol': 'BTCUSDT', 'price': 81580.53, 'ts': '2026-10-17 11:43:06'}
message value {'symbol': 'ETHUSDT', 'price': 2435.13, 'ts': '2026-10-17 11:43:06'}
message value {'symbol': 'BTCUSDT', 'price': 81580.53, 'ts': '2026-10-17 11:43:08'}
message value {'symbol': 'ETHUSDT', 'price': 2435.13, 'ts': '2026-10-17 11:43:08'}
message value {'symbol': 'BTCUSDT', 'price': 81585.77, 'ts': '2026-10-17 11:43:11'}
message value {'symbol': 'ETHUSDT', 'price': 2435.14, 'ts': '2026-10-17 11:43:11'}
message value {'symbol': 'BTCUSDT', 'price': 81585.76, 'ts': '2026-10-17 11:43:13'}
message value {'symbol': 'ETHUSDT', 'price': 2435.13, 'ts': '2026-10-17 11:43:13'}
message value {'symbol': 'BTCUSDT', 'price': 81585.76, 'ts': '2026-10-17 11:43:16'}
message value {'symbol': 'ETHUSDT', 'price': 2435.13, 'ts': '2026-10-17 11:43:16'}
loaded 10 rows
message value {'symbol': 'BTCUSDT', 'price': 81585.77, 'ts': '2026-10-17 11:43:19'}
message value {'symbol': 'ETHUSDT', 'price': 2435.13, 'ts': '2026-

In [12]:
cur.execute('SELECT COUNT(*) FROM crypto_prices')
print(cur.fetchone())

cur.execute('SELECT * FROM crypto_prices ORDER BY event_ts DESC LIMIT 10')
for row in cur.fetchall():
    print(row)

(370,)
('BTCUSDT', Decimal('82975.38000000'), datetime.datetime(2026, 10, 11, 10, 47, 56), 0, 368)
('ETHUSDT', Decimal('2498.59000000'), datetime.datetime(2026, 10, 11, 10, 47, 56), 0, 369)
('BTCUSDT', Decimal('82975.38000000'), datetime.datetime(2026, 10, 11, 10, 47, 52), 0, 364)
('BNBUSDT', Decimal('747.09000000'), datetime.datetime(2026, 10, 11, 10, 47, 52), 0, 366)
('ETHUSDT', Decimal('2499.01000000'), datetime.datetime(2026, 10, 11, 10, 47, 52), 0, 365)
('SOLUSDT', Decimal('109.36000000'), datetime.datetime(2026, 10, 11, 10, 47, 52), 0, 367)
('BTCUSDT', Decimal('82975.38000000'), datetime.datetime(2026, 10, 11, 10, 47, 49), 0, 360)
('BNBUSDT', Decimal('747.08000000'), datetime.datetime(2026, 10, 11, 10, 47, 49), 0, 362)
('SOLUSDT', Decimal('109.35000000'), datetime.datetime(2026, 10, 11, 10, 47, 49), 0, 363)
('ETHUSDT', Decimal('2499.01000000'), datetime.datetime(2026, 10, 11, 10, 47, 49), 0, 361)
